In [0]:
from pyspark.sql.functions import col, trim, lower, coalesce, current_timestamp

CATALOG = "workspace"
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

# Customers
customers = (spark.read.table(f"{CATALOG}.bronze.customers")
  .selectExpr(
    "cast(customer_id as int) customer_id",
    "trim(customer_fname) customer_fname",
    "trim(customer_lname) customer_lname",
    "lower(trim(customer_email)) customer_email",
    "trim(customer_street) customer_street",
    "trim(customer_city) customer_city",
    "trim(customer_state) customer_state",
    "trim(customer_zipcode) customer_zipcode"
    )
  ).dropDuplicates(["customer_id"])
customers.write.mode("overwrite").option("overwriteSchema","true").saveAsTable(f"{CATALOG}.silver.customers")

# Departments
departments = (spark.read.table(f"{CATALOG}.bronze.departments")
  .selectExpr("cast(department_id as int) department_id", "trim(department_name) department_name" )
  ).dropDuplicates(["department_id"])
departments.write.mode("overwrite").option("overwriteSchema","true").saveAsTable(f"{CATALOG}.silver.departments")

# Categories
categories = (spark.read.table(f"{CATALOG}.bronze.categories")
  .selectExpr(
    "cast(category_id as int) category_id",
    "cast(category_department_id as int) category_department_id",
    "trim(category_name) category_name"
    )
  ).dropDuplicates(["category_id"])
categories.write.mode("overwrite").option("overwriteSchema","true").saveAsTable(f"{CATALOG}.silver.categories")

# Products (con Integridad Referencial a category/department y quarantine)
products = (spark.read.table(f"{CATALOG}.bronze.products")
  .selectExpr(
    "cast(product_id as int) product_id",
    "cast(product_category_id as int) product_category_id",
    "trim(product_name) product_name",
    "trim(product_description) product_description",
    "cast(product_price as decimal(12,2)) product_price",
    "trim(product_image) product_image"
    )
  ).dropDuplicates(["product_id"])

prod_join = (products.alias("p")
  .join(categories.alias("c"), col("p.product_category_id") == col("c.category_id"), "left")
  .join(departments.alias("d"), col("c.category_department_id") == col("d.department_id"), "left")
  .withColumn("is_orphan", col("c.category_id").isNull() | col("d.department_id").isNull()))

prod_conform = prod_join.filter(~col("is_orphan")).select(
  "p.product_id","p.product_category_id","p.product_name","p.product_description","p.product_price","p.product_image",
  col("c.category_id"), col("c.category_name"), col("d.department_id"), col("d.department_name"))

prod_quarantine = prod_join.filter(col("is_orphan"))
prod_quarantine.printSchema()
prod_conform.write.mode("overwrite").option("overwriteSchema","true").saveAsTable(f"{CATALOG}.silver.products")
prod_quarantine.write.mode("overwrite").option("overwriteSchema","true").saveAsTable(f"{CATALOG}.silver.products_quarantine")

# Orders
orders = (spark.read.table(f"{CATALOG}.bronze.orders")
  .selectExpr(
    "cast(order_id as int) order_id",
    "cast(order_customer_id as int) order_customer_id",
    "cast(order_date as timestamp) order_date",
    "trim(order_status) order_status"
    )
  ).dropDuplicates(["order_id"])
orders.write.mode("overwrite").option("overwriteSchema","true").saveAsTable(f"{CATALOG}.silver.orders")

# Order items
order_items = (spark.read.table(f"{CATALOG}.bronze.order_items")
  .selectExpr(
    "cast(order_item_id as int) order_item_id",
    "cast(order_item_order_id as int) order_item_order_id",
    "cast(order_item_product_id as int) order_item_product_id",
    "cast(order_item_quantity as int) order_item_quantity",
    "cast(order_item_subtotal as decimal(14,2)) order_item_subtotal",
    "cast(order_item_product_price as decimal(12,2)) order_item_product_price",
    )
  ).dropDuplicates(["order_item_id"])
order_items.write.mode("overwrite").option("overwriteSchema","true").saveAsTable(f"{CATALOG}.silver.order_items")